# 02 — Results
Run `uv run python -m forecast.run` first. Answers RQ1 (vs baselines), RQ2 (holidays & markdowns), RQ3 (drivers, SHAP).

In [ ]:
import json, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, shap, xgboost as xgb
from pathlib import Path
from forecast.data import load_panel
from forecast.features import build_features, model_rows, FEATURES
from forecast.splits import TEST_START, TEST_END, between
sns.set_theme(style='whitegrid'); OUT = Path('../outputs')
metrics = pd.read_csv(OUT/'metrics.csv'); preds = pd.read_parquet(OUT/'predictions.parquet')
json.loads((OUT/'best_params.json').read_text())

## RQ1 — XGBoost vs naive baselines and Random Forest

In [ ]:
tbl = metrics[metrics.model != 'stock_coverage'].set_index(['window','model'])[['WMAE','MAE','RMSE','MAE_holiday','MAE_normal']]
tbl.round(0)

In [ ]:
t = metrics[(metrics.window == 'test_2012') & (metrics.model != 'stock_coverage')].sort_values('WMAE')
best_base = t[t.model.str.startswith(('naive','mean'))].WMAE.min()
xgb_w = t.set_index('model').loc['xgb','WMAE']
print(f'XGBoost WMAE {xgb_w:,.0f} vs best baseline {best_base:,.0f} -> {1 - xgb_w/best_base:.1%} better')
t.plot.barh(x='model', y='WMAE', legend=False, figsize=(8,3), title='Test WMAE (lower is better)'); plt.show()

## RQ2 — Holidays and markdowns
Holiday vs normal error (both windows) and the with/without-markdown ablation.

In [ ]:
metrics[metrics.model.isin(['xgb','xgb_no_markdown','naive_last_year','naive_last'])].set_index(['window','model'])[['WMAE','MAE_holiday','MAE_normal']].round(0)

In [ ]:
bt = preds[preds.window == 'holiday_backtest_2011'].groupby('Date')[['Weekly_Sales','pred_xgb','pred_naive_last_year','pred_naive_last']].sum()/1e6
bt.plot(marker='o', figsize=(12,4), title='2011 holiday season: total sales ($M), actual vs forecasts'); plt.show()

## RQ3 — What drives the forecast (SHAP on a 5k-row test sample)

In [ ]:
df = model_rows(build_features(load_panel()))
test = df[between(df, TEST_START, TEST_END)].sample(5000, random_state=0)
model = xgb.XGBRegressor(); model.load_model(OUT/'xgb_final.json')
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(test[FEATURES])
shap.summary_plot(sv, test[FEATURES], max_display=15)

In [ ]:
imp = pd.Series(np.abs(sv).mean(0), index=FEATURES).sort_values(ascending=False)
imp.head(15).round(0)

## Stocking signal — forecast vs actual with the recommended upper bound

In [ ]:
cov = metrics[metrics.model == 'stock_coverage'][['window','coverage']]
print(cov.to_string(index=False))
fig, axes = plt.subplots(2, 2, figsize=(14,7), sharex=True)
for ax, (s, d) in zip(axes.flat, [(1, 1), (10, 92), (20, 38), (4, 72)]):
    p = preds[(preds.window == 'test_2012') & (preds.Store == s) & (preds.Dept == d)].sort_values('Date')
    ax.fill_between(p.Date, p.pred_xgb.clip(lower=0), p.stock_upper, alpha=.25, label='stock range')
    ax.plot(p.Date, p.Weekly_Sales, 'k-o', ms=3, label='actual'); ax.plot(p.Date, p.pred_xgb, label='XGBoost')
    ax.set_title(f'Store {s} / Dept {d}')
axes[0,0].legend(); fig.autofmt_xdate(); plt.show()